# Small-book quote-or-refuse — finetune (2026-10-13, v5)

Before running: **Runtime -> Change runtime type -> A100 if offered, else L4** (High-RAM not needed).

One session trains **two candidates** (same Qwen2.5-3B base) and downloads
**three zips**. When the picker opens, upload **both** train files at once:
`quote-refuse-2026-10-13.train-x2c.jsonl` and `quote-refuse-2026-10-13.train-x2.jsonl`.
Do not upload the bench file.

Slot A (x2c, seed 3407) exports q4_k_m + q8_0; Slot B (x2 content, seed
1234 — the seed control) exports q4_k_m only. The q8 zip is the quant A/B
from the pre-registered gate (`x2-trial-2026-10-14.md`, item 5): a wart
that vanishes at q8_0 is a quantization verdict, not a training verdict.

Watch the two **STOP-CHECK** lines after each training: the reply should be
one quote + one gloss line, then silence. A LONG OUTPUT verdict means tell
the agent before trusting that zip.

If run B crashes at load: Runtime -> Restart session, run cells 1-4 again
(upload both files again), then run the last cell only.

If a cell errors, copy the red text back to the agent; library drift happens.

v5: pins the toolchain that trained x2 (`unsloth==2026.10.5`,
`transformers==5.17.0`, fallback to latest), prints the versions actually
loaded, asserts each run loads its expected 167 cards — again after
responses-only prep, as the truncation canary — and gives each run its own
seed (A 3407, B 1234). v4's x2 won (battery 18-19/21, 0 loops) but the
live trial found refusal-margin warts on unmemorized forms, so Slot A
(x2c) swaps ~30 rows to perturbed forms, counts held; Slot B holds x2
content as the seed control.

Env trap (2026-10-14): a fresh Unsloth build defaulted the trainer to
max_length=1024 and silently dropped 152/167 rows (the assistant marker
was truncated away). The notebook passes SFTConfig(max_length=max_seq_length)
and adds the canary assert; watch the trainer banner — 'Num examples' must
be 167 for both runs.

In [ ]:
!pip install -q "unsloth==2026.10.5" "transformers==5.17.0" || pip install -q unsloth

In [ ]:
import importlib

for pkg in ["unsloth", "torch", "transformers", "trl", "peft"]:
    try:
        mod = importlib.import_module(pkg)
        print(pkg, getattr(mod, "__version__", "(no __version__)"))
    except Exception as exc:
        print(pkg, "import failed:", repr(exc))

In [ ]:
import gc
import os
import torch

RUNS = [
    {"model": "unsloth/Qwen2.5-3B-Instruct-bnb-4bit",
     "export": "smallbook_x2c",
     "data": "quote-refuse-2026-10-13.train-x2c.jsonl",
     "expected": 167,
     "seed": 3407},  # Slot A: x2 with ~30 answer rows swapped to perturbed forms
    {"model": "unsloth/Qwen2.5-3B-Instruct-bnb-4bit",
     "export": "smallbook_x2_seed1234",
     "data": "quote-refuse-2026-10-13.train-x2.jsonl",
     "expected": 167,
     "seed": 1234},  # Slot B: x2 content unchanged — seed control
    # If a model tag 404s: send the red text to the agent; do not invent tags.
]

missing = [r["data"] for r in RUNS if not os.path.exists(r["data"])]
if missing:
    from google.colab import files
    files.upload()  # select BOTH train files in one go
for r in RUNS:
    print(r["data"], "present:", os.path.exists(r["data"]))
assert all(os.path.exists(r["data"]) for r in RUNS), "upload both train files"

In [ ]:
def train_one(model_name, export_name, data_file, expected, seed, q8=False):
    gc.collect()
    torch.cuda.empty_cache()

    from datasets import load_dataset
    raw = load_dataset("json", data_files=data_file, split="train")
    print(len(raw), "cards loaded from", data_file)
    assert len(raw) == expected, (
        "row count mismatch: %s has %d cards, expected %d" % (data_file, len(raw), expected))

    from unsloth import FastLanguageModel
    from unsloth.chat_templates import train_on_responses_only
    from trl import SFTConfig, SFTTrainer
    from transformers import TrainingArguments

    max_seq_length = 4096
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=model_name,
        max_seq_length=max_seq_length,
        load_in_4bit=True,
    )
    model = FastLanguageModel.get_peft_model(
        model,
        r=16,
        lora_alpha=32,
        lora_dropout=0.0,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                        "gate_proj", "up_proj", "down_proj"],
        use_gradient_checkpointing="unsloth",
        random_state=seed,
    )

    dataset = raw.map(lambda row: {"text": tokenizer.apply_chat_template(
        row["messages"], tokenize=False, add_generation_prompt=False)})

    trainer = SFTTrainer(
        model=model,
        tokenizer=tokenizer,
        train_dataset=dataset,
        dataset_text_field="text",
        max_seq_length=max_seq_length,
        args=SFTConfig(
            per_device_train_batch_size=2,
            gradient_accumulation_steps=4,
            warmup_steps=5,
            max_steps=120,
            max_length=max_seq_length,
            learning_rate=2e-4,
            fp16=not torch.cuda.is_bf16_supported(),
            bf16=torch.cuda.is_bf16_supported(),
            logging_steps=5,
            optim="adamw_8bit",
            weight_decay=0.01,
            lr_scheduler_type="linear",
            seed=seed,  # per-run seed (RUNS); SFTConfig subclasses TrainingArguments
            output_dir="outputs",
            report_to="none",
        ),
    )
    # Train on the model's ANSWERS only — mask the system prompt and the
    # passages. v1 trained on the whole text and never learned to stop.
    trainer = train_on_responses_only(
        trainer,
        instruction_part="<|im_start|>user\n",
        response_part="<|im_start|>assistant\n",
    )
    # Truncation canary: a bad max_length silently drops rows whose
    # assistant marker was cut (152/167 were lost on 2026-10-14).
    print("rows after responses-only prep:", data_file, len(trainer.train_dataset), "of", expected)
    assert len(trainer.train_dataset) == expected, (
        "truncation canary: %s kept only %d of %d rows through dataset prep" % (
            data_file, len(trainer.train_dataset), expected))
    trainer.train()

    # Stop check BEFORE exporting: a short sane reply, then silence.
    FastLanguageModel.for_inference(model)
    prompt_part = dataset[0]["text"].split("<|im_start|>assistant\n")[0] + "<|im_start|>assistant\n"
    inputs = tokenizer([prompt_part], return_tensors="pt").to("cuda")
    out = model.generate(**inputs, max_new_tokens=250, do_sample=False)
    reply_ids = out[0][inputs["input_ids"].shape[1]:]
    print("STOP-CHECK reply:\n", tokenizer.decode(reply_ids, skip_special_tokens=False)[:500])
    print("STOP-CHECK verdict:",
          "looks sane" if reply_ids.shape[0] < 250 else
          "LONG OUTPUT - tell the agent before trusting this zip")

    model.save_pretrained_gguf(export_name, tokenizer, quantization_method="q4_k_m")
    import glob, shutil
    from google.colab import files as colab_files
    path = sorted(glob.glob(export_name + "_gguf/*.gguf"))[0]
    print("gguf:", path)
    shutil.make_archive(export_name + "_gguf", "zip", root_dir=path.rsplit("/", 1)[0])
    colab_files.download(export_name + "_gguf.zip")
    print("done:", export_name)

    if q8:
        # Quant A/B, pre-registered gate item 5 (x2-trial-2026-10-14.md):
        # the x2c warts must persist at q8_0; one that vanishes only at q4
        # is a quantization verdict, not a training verdict. Slot A only.
        # A failure here must not kill the session — the q4 zip above is safe.
        try:
            model.save_pretrained_gguf(export_name + "_q8", tokenizer,
                                       quantization_method="q8_0")
            q8_path = sorted(glob.glob(export_name + "_q8_gguf/*.gguf"))[0]
            print("q8 gguf:", q8_path)
            shutil.make_archive(export_name + "_q8_gguf", "zip",
                                root_dir=q8_path.rsplit("/", 1)[0])
            colab_files.download(export_name + "_q8_gguf.zip")
            print("done:", export_name + "_q8_gguf.zip")
        except Exception as exc:
            print("WARNING: q8_0 export/zip/download failed; the q4 zip is",
                  "already safe, session continues:", repr(exc))

In [ ]:
# Run A: x2c — 167 cards, seed 3407 (~20 min A100, ~30-45 min L4).
# Exports q4_k_m + q8_0: the q8 zip is the quant A/B from the
# pre-registered gate (x2-trial-2026-10-14.md).
train_one(RUNS[0]["model"], RUNS[0]["export"], RUNS[0]["data"],
          RUNS[0]["expected"], RUNS[0]["seed"], q8=True)

In [ ]:
# Run B: x2 content unchanged, seed 1234 — the seed control; 167 cards.
# q4_k_m only. Same session; memory is freed above.
train_one(RUNS[1]["model"], RUNS[1]["export"], RUNS[1]["data"],
          RUNS[1]["expected"], RUNS[1]["seed"])